In [1]:
!pip install -U torch torchvision torchaudio xformers --index-url https://download.pytorch.org/whl/cu128
!pip install -U unsloth
!pip install -U transformers datasets
!pip install -U trl

In [2]:
import torch

# Check if a GPU accelerator is actively detected by PyTorch
if torch.cuda.is_available() or (hasattr(torch, 'accelerator') and torch.accelerator.is_available()):
    print("✅ GPU detected successfully!")
    print("Device Name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Accelerator")
else:
    print("❌ NO GPU DETECTED. Please check your Colab runtime settings.")


✅ GPU detected successfully!
Device Name: Tesla T4


In [3]:
from dataclasses import dataclass
from typing import Optional,List
from datasets import load_dataset
from unsloth import FastLanguageModel
from peft import PeftModel
from trl import SFTTrainer,SFTConfig

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [4]:
@dataclass
class FineTuneConfig:
  model_name: str
  load_in_4bit: bool = True
  max_seq_length: int = 4096
  dtype: Optional[str] = None  # None = auto

  # dataset
  dataset_name: str = "tatsu-lab/alpaca"
  split: str = "train"
  seed: int = 3407

  # training
  output_dir: str = "outputs"
  lora_save_path: str = "lora_adapters"
  per_device_bs: int = 2
  grad_acc_steps: int = 4
  epochs: int = 1
  lr: float = 2e-5
  warmup_ratio: float = 0.1
  logging_steps: int = 10
  packing: bool = True

  # lora
  lora_r: int = 32
  lora_alpha: int = 32
  lora_dropout: float = 0.0
  target_modules: List[str] = None
  use_gc: bool = False

  # save merged model
  save_merged: bool = False
  merged_save_path: str = "merged_fp16_model"



In [14]:
class UnslothFineTuner:
  def __init__(self,cfg:FineTuneConfig):
    self.cfg = cfg
    if self.cfg.target_modules is None:
      self.cfg.target_modules = [
          "q_proj","k_proj","v_proj","o_proj",
          "gate_proj","up_proj","down_proj"]
    self.model = None
    self.tokenizer = None

  def load_model(self):
    print(f"✅ Loading model: {self.cfg.model_name}")

    self.model, self.tokenizer = FastLanguageModel.from_pretrained(
        model_name = self.cfg.model_name,
        load_in_4bit = self.cfg.load_in_4bit,
        dtype = self.cfg.dtype,
        # target_modules = self.cfg.target_modules,
        # use_gc = self.cfg.use_gc
    )

    self.model = FastLanguageModel.get_peft_model(
        self.model,
        r = self.cfg.lora_r,
        lora_alpha = self.cfg.lora_alpha,
        lora_dropout = self.cfg.lora_dropout,
        target_modules = self.cfg.target_modules,
        bias = "none",
    )
    self.model.print_trainable_parameters()
    print("Is PEFT model?", isinstance(self.model, PeftModel))
    print("Device:", next(self.model.parameters()).device, " | Dtype:", next(self.model.parameters()).dtype)


  def load_dataset(self):
    print(f"✅ Loading dataset: {self.cfg.dataset_name}")
    dataset = load_dataset(self.cfg.dataset_name,split=self.cfg.split)
    dataset = dataset.shuffle(seed=self.cfg.seed)

    print("Columns:", dataset.column_names)
    return dataset

  def _eos(self):
        return self.tokenizer.eos_token or ""

  def _alpaca_prompt(self):
        return """Below is an instruction that describes a task, paired with an input that provides further context.
            Write a response that appropriately completes the request.

            ### Instruction:
            {instruction}

            ### Input:
            {input}

            ### Response:
            {output}"""

  def _format_alpaca(self, batch):
    eos = self._eos()
    prompt = self._alpaca_prompt()
    texts = []

    inp_list = batch.get("input", [""]*len(batch['instruction']))
    for ins, inp, out in zip(batch["instruction"], inp_list, batch["output"]):
        texts.append(prompt.format(
            instruction=ins or "",
            input=inp or "",
            output=out or ""
        ) + eos)
    return {"text": texts}

  def _format_dolly(self, batch):
          EOS = self._eos()
          prompt = self._alpaca_prompt()
          ctx_list = batch.get("context", [""] * len(batch["instruction"]))
          texts = []
          for ins, ctx, resp in zip(batch["instruction"], ctx_list, batch["response"]):
              texts.append(prompt.format(
                  instruction=ins or "",
                  input=ctx or "",
                  output=resp or ""
              ) + EOS)
          return {"text": texts}

  def _format_sharegpt(self, batch):
          EOS = self._eos()
          texts = []

          conv_key = None
          if "conversations" in batch:
              conv_key = "conversations"
          elif "messages" in batch:
              conv_key = "messages"
          else:
              raise ValueError("ShareGPT format needs 'conversations' or 'messages' column.")

          for conv in batch[conv_key]:
              if conv is None:
                  texts.append("" + EOS)
                  continue

              turns = []
              for t in conv:
                  if isinstance(t, dict) and "from" in t and "value" in t:
                      role, content = t["from"], t["value"]
                  elif isinstance(t, dict) and "role" in t and "content" in t:
                      role, content = t["role"], t["content"]
                  else:
                      continue
                  turns.append((role, content))

              chat = ""
              for role, content in turns:
                  if role in ["human", "user"]:
                      chat += f"### User:\n{content}\n\n"
                  else:
                      chat += f"### Assistant:\n{content}\n\n"

              texts.append(chat.strip() + EOS)

          return {"text": texts}

  def _format_text(self, batch):
        EOS = self._eos()
        if "text" not in batch:
            raise ValueError("Expected a 'text' column.")
        return {"text": [(t or "") + EOS for t in batch["text"]]}


  def _format_pharma_custom(self, batch):
        """
        For sunny199/pharma-instruction-data OR pharma_instruction_data
        Supports common patterns:
        - instruction,input,output
        - question,answer
        - prompt,completion
        - input,output
        """
        EOS = self._eos()
        cols = set(batch.keys())

        if {"instruction", "output"}.issubset(cols):
            return self._format_alpaca(batch)

        if {"question", "answer"}.issubset(cols):
            texts = []
            for q, a in zip(batch["question"], batch["answer"]):
                texts.append(f"### Question:\n{q or ''}\n\n### Answer:\n{a or ''}" + EOS)
            return {"text": texts}

        if {"prompt", "completion"}.issubset(cols):
            texts = []
            for p, c in zip(batch["prompt"], batch["completion"]):
                texts.append(f"{p or ''}\n{c or ''}" + EOS)
            return {"text": texts}

        if {"input", "output"}.issubset(cols):
            prompt = self._alpaca_prompt()
            texts = []
            for i, o in zip(batch["input"], batch["output"]):
                texts.append(prompt.format(
                    instruction="Answer the following:",
                    input=i or "",
                    output=o or ""
                ) + EOS)
            return {"text": texts}

        raise ValueError(f"Pharma formatter can't infer columns: {sorted(list(cols))}")

  def format_dataset(self, ds):
        print("✅ Inferring dataset format...")

        name = self.cfg.dataset_name
        cols = set(ds.column_names)

        # known dataset mappings
        if name == "tatsu-lab/alpaca":
            print("✅ Format: ALPACA")
            return ds.map(self._format_alpaca, batched=True, remove_columns=ds.column_names)

        if name == "databricks/databricks-dolly-15k":
            print("✅ Format: DOLLY")
            return ds.map(self._format_dolly, batched=True, remove_columns=ds.column_names)

        if name == "anon8231489123/ShareGPT_Vicuna_unfiltered":
            print("✅ Format: SHAREGPT")
            return ds.map(self._format_sharegpt, batched=True, remove_columns=ds.column_names)

        if name == "OpenAssistant/oasst1":
            print("✅ Format: OASST (auto)")
            if "text" in cols:
                return ds.map(self._format_text, batched=True, remove_columns=ds.column_names)
            return ds.map(self._format_sharegpt, batched=True, remove_columns=ds.column_names)

        if name in ["sunny199/pharma-instruction-data", "pharma_instruction_data"]:
            print("✅ Format: PHARMA (custom)")
            return ds.map(self._format_pharma_custom, batched=True, remove_columns=ds.column_names)

        # fallback heuristics
        if "text" in cols:
            print("✅ Format: TEXT (fallback)")
            return ds.map(self._format_text, batched=True, remove_columns=ds.column_names)
        if "conversations" in cols or "messages" in cols:
            print("✅ Format: CHAT (fallback)")
            return ds.map(self._format_sharegpt, batched=True, remove_columns=ds.column_names)

        if {"instruction", "output"}.issubset(cols):
            print("✅ Format: ALPACA-LIKE (fallback)")
            return ds.map(self._format_alpaca, batched=True, remove_columns=ds.column_names)

        raise ValueError(f"❌ Could not infer dataset format. Columns: {sorted(list(cols))}")

  def train(self, ds):
        print("✅ Starting training...")
        trainer = SFTTrainer(
            model=self.model,
            tokenizer=self.tokenizer,
            train_dataset=ds,
            dataset_text_field="text",
            packing=self.cfg.packing,
            args=SFTConfig(
                per_device_train_batch_size=self.cfg.per_device_bs,
                gradient_accumulation_steps=self.cfg.grad_acc_steps,
                num_train_epochs=self.cfg.epochs,
                learning_rate=self.cfg.lr,
                warmup_ratio=self.cfg.warmup_ratio,
                optim="adamw_8bit",
                logging_steps=self.cfg.logging_steps,
                seed=self.cfg.seed,
                output_dir=self.cfg.output_dir,
                report_to="none",
            ),
        )
        trainer.train()


  # ---------- Full pipeline ----------
  def run(self):
        self.load_model()
        raw_ds = self.load_dataset()
        ds = self.format_dataset(raw_ds)
        print("✅ Sample formatted text:\n", ds["text"][0][:800])
        self.train(ds)
        # self.save() # The save() method is not defined in the provided code.
        print("✅ Done!")

In [15]:
cfg = FineTuneConfig(
    model_name="unsloth/Phi-3-mini-4k-instruct-bnb-4bit",
    dataset_name="sunny199/pharma-instruction-data",
    split="train",
    lora_save_path="phi3_pharma_lora",
    output_dir="phi3_outputs",
    epochs=1,
    per_device_bs=2,
    grad_acc_steps=4,
    lr=2e-5,
    max_seq_length=4096,
    save_merged=False,  # True if you want merged fp16 model
)

trainer = UnslothFineTuner(cfg)
trainer.run()

✅ Loading model: unsloth/Phi-3-mini-4k-instruct-bnb-4bit
==((====))==  Unsloth 2026.9.4: Fast Mistral patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth 2026.9.4 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


trainable params: 59,768,832 || all params: 3,880,848,384 || trainable%: 1.5401
Is PEFT model? True
Device: cuda:0  | Dtype: torch.float16
✅ Loading dataset: sunny199/pharma-instruction-data


pharma_instruction_data.csv:   0%|          | 0.00/2.47k [00:00<?, ?B/s]

...

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2 | Num Epochs = 1 | Total steps = 1
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 59,768,832 of 3,880,848,384 (1.54% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss


Unsloth: Restored added_tokens_decoder metadata in phi3_outputs/checkpoint-1/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in phi3_outputs/checkpoint-1.


✅ Done!
